# Save, Load, Inference Smoke Test

This notebook checks the deployment path used by the training jobs: instantiate a model variant, save a plain base-model `state_dict`, load it into a fresh model, and run inference. It supports both the compact model and DELight pairwise variants through `reconstruction_model.models`.

In [ ]:
from pathlib import Path
import json
import re
import subprocess
import sys
from dataclasses import fields
from typing import Any

import torch

repo_candidates = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next(path for path in repo_candidates if (path / "reconstruction_model").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from reconstruction_model.models import create_model, load_model_objects

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Repo root: {REPO_ROOT}")
print(f"Device: {DEVICE}")

In [ ]:
def torch_load_state_dict(path: Path):
    try:
        state_dict = torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        state_dict = torch.load(path, map_location="cpu")
    if not isinstance(state_dict, dict):
        raise TypeError(f"Expected a state dict at {path}, got {type(state_dict)!r}")
    if state_dict and all(key.startswith("_orig_mod.") for key in state_dict):
        state_dict = {key.removeprefix("_orig_mod."): value for key, value in state_dict.items()}
    return state_dict


def model_config_from_dict(variant: str, config_dict: dict[str, Any]):
    _, config_cls = load_model_objects(variant)
    valid_fields = {field.name for field in fields(config_cls)}
    filtered = {key: value for key, value in config_dict.items() if key in valid_fields}

    # Condor run configs may contain absolute cache paths from the execute node.
    # If they are stale after transfer, use the package-local defaults instead.
    for path_key in ("pairwise_feats_path",):
        if path_key in filtered and not Path(filtered[path_key]).exists():
            filtered.pop(path_key)

    return config_cls(**filtered)


def instantiate_model(variant: str, config_dict: dict[str, Any] | None = None):
    if config_dict is None:
        model, config = create_model(variant)
    else:
        config = model_config_from_dict(variant, config_dict)
        model_cls, _ = load_model_objects(variant)
        model = model_cls(config)
    return model, config


def load_model_for_inference(checkpoint_path: Path, run_config_path: Path | None = None):
    checkpoint_path = Path(checkpoint_path)
    if run_config_path is None:
        candidate = checkpoint_path.parent / "run_config.json"
        run_config_path = candidate if candidate.exists() else None

    if run_config_path is not None:
        run_config = json.loads(Path(run_config_path).read_text())
        variant = run_config.get("model_variant", "current_compact")
        config_dict = run_config.get("model_config", {})
    else:
        run_config = {}
        variant = "current_compact"
        config_dict = None

    model, config = instantiate_model(variant, config_dict)
    state_dict = torch_load_state_dict(checkpoint_path)
    model.load_state_dict(state_dict, strict=True)
    model.to(DEVICE)
    model.eval()
    return model, config, variant, run_config


def smoke_input_for_config(config, seq_len: int | None = None):
    n_channels = getattr(config, "n_channels", 56)
    max_seq_len = getattr(config, "max_seq_len", 65536)
    patch_len = getattr(config, "patch_len", 128)
    if seq_len is None:
        seq_len = min(max_seq_len, max(patch_len * 4, 512))
    seq_len = max(seq_len, patch_len)
    return torch.randn(1, n_channels, seq_len, device=DEVICE)


def run_inference_smoke(model, config):
    with torch.inference_mode():
        outputs = model(smoke_input_for_config(config))
    if not isinstance(outputs, (tuple, list)):
        outputs = (outputs,)
    shapes = [tuple(output.shape) for output in outputs]
    print(f"Output shapes: {shapes}")
    return outputs

## Synthetic Pairwise Save/Load Test

This uses a tiny `pairwise_channel_masking` config so the notebook runs quickly. It verifies that a plain state dict can be saved, loaded into a fresh model, and used for inference.

In [ ]:
torch.manual_seed(123)

smoke_dir = REPO_ROOT / "artifacts" / "notebook_inference_smoke"
smoke_dir.mkdir(parents=True, exist_ok=True)
smoke_checkpoint = smoke_dir / "pairwise_channel_masking_smoke.pt"

smoke_overrides = {
    "max_seq_len": 512,
    "patch_len": 64,
    "patch_stride": 64,
    "d_model": 32,
    "d_ff": 64,
    "n_head": 4,
    "n_temporal_blocks": 1,
    "n_channel_blocks": 1,
    "dropout_transformer": 0.0,
    "dropout_ffn": 0.0,
    "mask_channel_prob": 0.0,
}

model, config = create_model("pairwise_channel_masking", **smoke_overrides)
model.to(DEVICE)
model.eval()
run_inference_smoke(model, config)

torch.save(model.state_dict(), smoke_checkpoint)
print(f"Saved smoke checkpoint: {smoke_checkpoint}")

fresh_model, fresh_config = instantiate_model("pairwise_channel_masking", smoke_overrides)
fresh_model.load_state_dict(torch_load_state_dict(smoke_checkpoint), strict=True)
fresh_model.to(DEVICE)
fresh_model.eval()
run_inference_smoke(fresh_model, fresh_config)

## Optional: Load Latest Trained Checkpoint

This searches `artifacts/**/reconstruction_model_*.pt`, reads the neighboring `run_config.json` when available, instantiates the recorded model variant, and runs a short dummy inference pass.

In [ ]:
def checkpoint_step(path: Path):
    match = re.search(r"reconstruction_model_(\d+)\.pt$", path.name)
    return int(match.group(1)) if match else -1


candidate_checkpoints = sorted(
    (REPO_ROOT / "artifacts").glob("**/reconstruction_model_*.pt"),
    key=lambda path: (path.stat().st_mtime, checkpoint_step(path)),
)

if not candidate_checkpoints:
    print("No trained checkpoints found under artifacts/ yet.")
else:
    latest_checkpoint = candidate_checkpoints[-1]
    model, config, variant, run_config = load_model_for_inference(latest_checkpoint)
    print(f"Loaded checkpoint: {latest_checkpoint}")
    print(f"Variant: {variant}")
    print(f"Config: {config}")
    run_inference_smoke(model, config)

## Optional: Download Latest Remote Checkpoint

Set `REMOTE_RUN_DIRECTORY` to a run directory published by training. The cell downloads `latest.json`, the model-only checkpoint, and `run_config.json` through XRootD before running inference.

In [ ]:
def xrdcp(source: str, destination: Path):
    destination.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["xrdcp", "--nopbar", "--force", source, str(destination)],
        check=True,
    )
    return destination


def download_latest_remote_run(remote_directory: str):
    download_dir = REPO_ROOT / "artifacts" / "remote_inference_download"
    latest_path = xrdcp(
        f"{remote_directory.rstrip('/')}/latest.json",
        download_dir / "latest.json",
    )
    latest = json.loads(latest_path.read_text())
    files = latest["files"]
    model_name = next(name for name in files if name.startswith("reconstruction_model_"))
    model_path = xrdcp(files[model_name], download_dir / model_name)
    config_path = xrdcp(files["run_config.json"], download_dir / "run_config.json")
    return model_path, config_path, latest


REMOTE_RUN_DIRECTORY = None  # e.g. root://ceph-node-j.etp.kit.edu//dwong/training_runs/pairwise_l40s_12345

if REMOTE_RUN_DIRECTORY is None:
    print("Set REMOTE_RUN_DIRECTORY to download a published training checkpoint.")
else:
    remote_model_path, remote_config_path, latest = download_latest_remote_run(
        REMOTE_RUN_DIRECTORY
    )
    model, config, variant, run_config = load_model_for_inference(
        remote_model_path,
        remote_config_path,
    )
    print(f"Downloaded step {latest['step']} at epoch {latest['epoch']:.3f}")
    print(f"Variant: {variant}")
    run_inference_smoke(model, config)

## Optional: Load Latest W&B Artifact

Set `WANDB_ARTIFACT_REFERENCE` to an artifact such as `entity/DELight_Reconstruction_Pairwise_Full/pairwise_channel_masking_l40s_12345-checkpoint:latest`. The logged artifact contains both inference and resume checkpoints plus `run_config.json`.

In [ ]:
import wandb


def download_wandb_checkpoint_artifact(reference: str):
    artifact = wandb.Api().artifact(reference, type="model-checkpoint")
    download_dir = Path(
        artifact.download(
            root=REPO_ROOT / "artifacts" / "wandb_inference_download"
        )
    )
    model_paths = sorted(download_dir.glob("reconstruction_model_*.pt"))
    if not model_paths:
        raise FileNotFoundError("Artifact contains no model-only checkpoint")
    return model_paths[-1], download_dir / "run_config.json", artifact


WANDB_ARTIFACT_REFERENCE = None

if WANDB_ARTIFACT_REFERENCE is None:
    print("Set WANDB_ARTIFACT_REFERENCE to download a W&B checkpoint artifact.")
else:
    wandb_model_path, wandb_config_path, artifact = (
        download_wandb_checkpoint_artifact(WANDB_ARTIFACT_REFERENCE)
    )
    model, config, variant, run_config = load_model_for_inference(
        wandb_model_path,
        wandb_config_path,
    )
    print(f"Downloaded artifact: {artifact.qualified_name}")
    print(f"Variant: {variant}")
    run_inference_smoke(model, config)